In [1]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import os

from Params import Config
from Net import CAFNet
from Dataset import DeapDataset, MultimodalDataset
from Dataloader_Excel import read_excel_eeg, read_excel_nirs
from train_validate import train
from SPL_Loss import MySpl_loss
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from Metrics import metrics

if __name__ == '__main__':
	
	torch.manual_seed(42)
	torch.cuda.manual_seed(42)
	device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

	if not os.path.exists('Results'):
		os.mkdir('Results')

	for subject in range(29):
		subject += 1
		print('training on:', device)

		args = Config().parse()

		mode = 0
		eeg, labels = read_excel_eeg(subject, mode)
		nirs = read_excel_nirs(subject, mode)
		dataset = MultimodalDataset(eeg, nirs, labels)

		acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []

		train_size = int(0.6 * len(eeg)) # 60 trials/subject, 60/40 for training/testing
		eval_size = len(eeg) - train_size
		
		dataset = torch.utils.data.TensorDataset(eeg, nirs, labels)
		train_dataset, eval_dataset = torch.utils.data.random_split(dataset, [train_size, eval_size])
		train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
		eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=args.batch_size, shuffle=False)
		
		model = CAFNet(eeg_dim=4000, nirs_dim=200, 
					hidden_size=64, num_layers=1, 
					dim=64, heads=1, dim_head=64, 
					mlp_dim=128, num_classes=2, dropout=0.3).to(device, dtype=torch.float64)
		
		optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=1e-5)
		scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
		
		# cross entropy loss and self-paced learning (spl) loss
		# len(train_loader.dataset) = n_trial, e.g. 60 len(train_loader) = n_batch, e.g. n_trial // batch_size
		criterion1 = nn.CrossEntropyLoss()
		criterion2 = MySpl_loss(n_samples=len(train_loader.dataset), batch_size=args.batch_size, alpha=args.alpha, beta=args.beta, spl_lambda=args.spl_lambda, spl_gamma=args.spl_gamma)
		criterions = [criterion1, criterion2]

		_, _, acc, precision, recall, f1, kappa, _, _ = train(model, criterions, optimizer, scheduler, train_loader, eval_loader, device, args.epochs, args.alpha, args.beta, args.task)

		acc_list.extend(acc)
		precision_list.extend(precision)
		recall_list.extend(recall)
		f1_list.extend(f1)
		kappa_list.extend(kappa)

		metrics(subject, args.log_name, args.log_mode, acc_list, precision_list, recall_list, f1_list, kappa_list)

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


training on: cuda:0
1 EEG torch.Size([60, 30, 4000]) Labels torch.Size([60])
1 fNIRS torch.Size([60, 36, 200])
train_loss = 1.0501 train_acc = 0.5556 valid_loss = 0.7920 valid_acc = 0.6250
train_loss = 0.7995 train_acc = 0.6389 valid_loss = 0.7701 valid_acc = 0.5833
train_loss = 0.7968 train_acc = 0.6944 valid_loss = 0.8411 valid_acc = 0.6250
train_loss = 0.7002 train_acc = 0.7222 valid_loss = 0.8096 valid_acc = 0.6250
train_loss = 0.5976 train_acc = 0.7778 valid_loss = 0.8235 valid_acc = 0.5833
train_loss = 0.5850 train_acc = 0.7500 valid_loss = 0.9106 valid_acc = 0.6667
train_loss = 0.4913 train_acc = 0.7778 valid_loss = 1.1120 valid_acc = 0.6250
train_loss = 0.3895 train_acc = 0.8611 valid_loss = 1.4751 valid_acc = 0.6250
train_loss = 0.5804 train_acc = 0.8333 valid_loss = 1.5125 valid_acc = 0.6250
train_loss = 0.5403 train_acc = 0.7778 valid_loss = 1.3262 valid_acc = 0.6250
train_loss = 0.5430 train_acc = 0.8333 valid_loss = 1.1412 valid_acc = 0.6250
train_loss = 0.4039 train_acc =